# Bronze Incremental

# Step 1: Import and Setup

This cell imports the PySpark and Delta helpers used in the notebook, switches to the correct catalog, and makes sure the Bronze schema exists before we start loading data.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *
from datetime import datetime
import uuid

In [0]:
spark.sql("USE CATALOG dbw_data_prj")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Step 2: Bronze Control Table
This table stores the watermark for each source table.

It helps the pipeline remember:

- the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in the latest run

This is what makes the Bronze load incremental and rerun-safe.

In [0]:
spark.sql(
        """
          CREATE TABLE IF NOT EXISTS dbw_data_prj.bronze.ingestion_control
          (
              layer_name STRING,
              table_name STRING,
              current_timestamp TIMESTAMP,
              currnet_pk INT,
              last_successful_timestamp TIMESTAMP,
              last_successful_pk INT,
              last_run_id STRING,
              rows_affected BIGINT,
              run_status STRING,
              updated_at TIMESTAMP
          )
          USING DELTA
        """
        )

# Step 3: Source Table Configuration
This cell defines which source tables will be loaded into Bronze and which columns should be used as:

- primary key
- timestamp / watermark column

It also creates a unique `bronze_run_id` for the current pipeline run.

In [0]:
table_config = {
    "orders": {"current_timestamp": "updated_at", "current_pk": "order_id"},
    "products": {"current_timestamp": "updated_at", "current_pk": "product_id"},
    "payments": {"current_timestamp": "processed_at", "current_pk": "payment_id"}
}

bronze_run_id = uuid.uuid4()
print("Current Bronze Run ID: ",bronze_run_id)

# Step 4: Helper Functions
This cell contains reusable functions:

- `get_last_successful_watermark()` reads the last processed watermark from the control table
- `upsert_bronze_control()` updates the control table after a successful Bronze load

These functions keep the main load logic cleaner and easier to understand.





In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl = (
        spark.table("dbw_data_prj.bronze.ingestion_control")
        .filter(
            (col("layer_name") == "bronze")
            & (col("table_name") == table_name)
            & (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None
    else:
        return rows[0]["last_successful_timestamp"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, current_timestamp, currnet_pk, last_successful_timestamp, last_successful_pk, last_run_id, rows_affected):
    ctrl_df = spark.createDataFrame(
        [
            ("bronze", table_name, current_timestamp, currnet_pk, last_successful_timestamp, last_successful_pk, last_run_id, rows_affected, "success", datetime.utcnow())
        ],
        schema = """
            layer_name STRING,
            table_name STRING,
            current_timestamp TIMESTAMP,
            currnet_pk INT,
            last_successful_timestamp TIMESTAMP,
            last_successful_pk INT,
            last_run_id STRING,
            rows_affected BIGINT,
            run_status STRING,
            updated_at TIMESTAMP
            """
    )
    deltaTable = DeltaTable.forName(spark, "dbw_data_prj.bronze.ingestion_control")
    (
        deltaTable.alias("t")
        .merge(ctrl_df.alias("s"), "t.table_name = s.table_name AND t.layer_name = s.layer_name")
        .whenMatchedUpdate(
            set = {
                "current_timestamp": col("s.current_timestamp"),
                "currnet_pk": col("s.currnet_pk"),
                "last_successful_timestamp": col("s.last_successful_timestamp"),
                "last_successful_pk": col("s.last_successful_pk"),
                "last_run_id": col("s.last_run_id"),
                "rows_affected": col("s.rows_affected"),
                "run_status": col("s.run_status"),
                "updated_at": col("s.updated_at")
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

# Step 5 - Bronze incremental load loop
This is the main Bronze logic.

For each table, the notebook:

- reads the last watermark
- reads the source SQL table
- filters only new / changed rows
- adds Bronze audit columns
- appends the rows into the Bronze Delta table
- updates the control table

This is the core incremental loading logic.

In [0]:
for table_name, cfg in table_config.items():
    current_timestamp_col = cfg["current_timestamp"]
    current_pk_col = cfg["current_pk"]
    source_table = f"`novacart-catalog`.dbo.{table_name}"
    target_table = f"dbw_data_prj.bronze.{table_name}_raw"

    last_successful = get_last_successful_watermark(table_name)
    if last_successful is None:
        last_successful_timestamp, last_successful_pk = None, None
    else:
        last_successful_timestamp, last_successful_pk = last_successful

    print(f"Processing.....{table_name}")
    print(f"last_successful_timestamp:{last_successful_timestamp}")
    print(f"last_successful_pk:{last_successful_pk}")

    source_df = (
        spark.read.table(source_table)
        .withColumn("current_timestamp", col(current_timestamp_col))
        .withColumn("current_pk", col(current_pk_col))
    )

    if last_successful_timestamp is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (col("current_timestamp") > lit(last_successful_timestamp)) |
            ((col("current_timestamp") == lit(last_successful_timestamp)) & (col("current_pk") > lit(last_successful_pk)))
        )

    rows_to_load = (
        rows_to_load.withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_run_id", lit(str(bronze_run_id)))
        .withColumn("bronze_source_table", lit(source_table))
    )

    row_count = rows_to_load.count()
    print(f"{table_name} rows to load: {row_count}")

    if row_count == 0:
        print(f"No new rows for {table_name}")
        upsert_bronze_control(
            table_name,
            None,
            None,
            last_successful_timestamp,
            last_successful_pk,
            bronze_run_id,
            0
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_timestamp_row = rows_to_load.agg(max("current_timestamp").alias("max_timestamp")).collect()[0]
    max_timestamp = max_timestamp_row["max_timestamp"]

    max_pk_row = (
        rows_to_load.filter(col("current_timestamp") == lit(max_timestamp))
        .agg(max("current_pk").alias("max_pk"))
        .collect()[0]
    )
    max_pk = max_pk_row["max_pk"]

    upsert_bronze_control(
        table_name,
        max_timestamp,
        max_pk,
        max_timestamp,
        max_pk,
        bronze_run_id,
        row_count
    )
    print(f"Finished {table_name} with {row_count} rows")

# Step 6: Quick validation
This final cell prints the Bronze row counts and displays the control table so you can verify that the incremental logic is working correctly.

In [0]:
print("Orders Bronze Count: ", spark.sql("select count(*) from dbw_data_prj.bronze.orders_raw").collect()[0][0])
print("Products Bronze Count: ", spark.sql("select count(*) from dbw_data_prj.bronze.products_raw").collect()[0][0])
print("Payments Bronze Count: ", spark.sql("select count(*) from dbw_data_prj.bronze.payments_raw").collect()[0][0])

display(spark.sql("select * from dbw_data_prj.bronze.ingestion_control").orderBy("table_name"))